# Qari-OCR-0.3 Test: Arabic-Specialized Vision Model

This notebook tests **Qari-OCR-0.3 (2B)**, a model specifically optimized for Arabic document understanding and OCR.

### 1. Setup

```bash
uv pip install openai pillow
```


In [1]:
import base64
import time
from openai import OpenAI
from PIL import Image
import io
import sys

# Point to the local server (LM Studio / Ollama)
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


img_path = "ollama_test_table.png"
base64_image = encode_image(img_path)

print("Setup complete. Ready to talk to Qari-OCR.")

Setup complete. Ready to talk to Qari-OCR.


### 2. Run Inference: Document Parsing (STREAMING)

Since Qari is Arabic-centric, we will use a direct Arabic prompt to see how it handles the tables and titles.


In [2]:
print("Requesting streaming inference from Qari-OCR-0.3...\n")
start_time = time.time()

response = client.chat.completions.create(
    model="qari-ocr",  # LM Studio uses the loaded model
    messages=[
        {
            "role": "system",
            "content": "أنت مساعد خبير في استخراج النصوص والجداول من الصور. قم بتحويل الصورة إلى تنسيق Markdown بدقة عالية جداً. حافظ على بنية الجداول واستخرج كل الكلمات بدقة.",
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "قم باستخراج كافة النصوص والجداول من هذه الصورة بتنسيق Markdown.",
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                },
            ],
        },
    ],
    max_tokens=4096,
    temperature=0,
    stream=True,
)

full_response = ""
print("--- Real-time Output ---")
for chunk in response:
    if chunk.choices[0].delta.content:
        content = chunk.choices[0].delta.content
        print(content, end="", flush=True)
        full_response += content

end_time = time.time()
print(f"\n\n--- Inference Finished in {end_time - start_time:.2f} seconds ---")

Requesting streaming inference from Qari-OCR-0.3...

--- Real-time Output ---
<h1><u>«الجنح</u></h1><br><table><thead><tr><th>النقط</th><th>الواجب</th><th>خصصها</th></tr></thead><tbody><tr><td>...</td><td>...</td><td>01</td></tr><tr><td>...</td><td>...</td><td>07</td></tr><tr><td>6</td><b>-</b> سيافة مركبة تحت تأثير الكحول <i>أو</i> تحت تأثير المواد المخدرة <u>الرقم</u> الترتيبي 208 و213 أدناه</tr><tr><td>...</td><b>-</b> رفض الخضوع للراتز المشار إليه في المادة 207 أدناه أو <i>للتحفقات</i> أو لاختبارات الكشف المنصوص عليها في المادتين 208 و213 أدناه</tr><tr><td>...</td><b>-</b> السائق الذي وجه إليه الأمر بالتوقيف وامتنع من تنفيذه <u>أولم</u> يحترم الأمر بتوقيف المركبة أو رفض سياقة مركبته أوالعمل على سياقها إلى <i>المحجز</i> أو رفض الامتثال للأوامر القانونية الصادرة إليه</td></tr><tr><td>...</td><b>-</b> 13</td></tr><tr><td>...</td><u>-</u> 18</tr></tbody></table><br><h2><i>«المخالفات</i></h2><br><table><thead><tr><th>النقط</th><th>الواجب</th><th>خصصها</th></tr></thead><tbody><tr><td>...

### 3. Display Rendered Result


In [ ]:
from IPython.display import display, Markdown

display(Markdown(full_response))